# preborn_pipeline — serverless fork of the Prod synthetic pipeline

Fork of `/Workspace/Shared/ADC-DB/Prod/Synthetic/pipeline` for PREBORN. Reuses
the shared **engine** (rekey/temporal/rarity/synth_io) verbatim, but replaces the
OMOP-specific cohort builder, drops the reference group + death base-rate logic,
and adds Phase B (`preborn_derive`). PREBORN has TWO generators (mother, infant);
this notebook runs ONE per invocation (widget `role`), then a final `derive`
invocation unifies + re-derives + validates.

Widgets:
* `role`   — mother | infant | derive
* `mode`   — full | fixture   (fixture = mock generate, no GPU/mostlyai)
* `stage`  — auto | pre | post   (per-role: pre=cohort+profile+manifest,
             post=verify gen_* → assemble → stg_*)
* `cohort_persons` — subsample size (fixture forces small)

Order for a full dev run:
1. role=mother stage=pre   → cohort_* in synth_preborn_m
2. (GPU) preborn_train role=mother  → gen_* + manifest
3. role=mother stage=post  → stg_* in synth_preborn_m
4–6. same for role=infant
7. role=derive  → synth_* finals in synth_preborn + final validation

In `mode=fixture`, steps 1–6 collapse: role=mother/infant stage=auto runs
cohort→mock→assemble in one go; then role=derive.

In [0]:
%run /Workspace/Shared/ADC-DB/Prod/Synthetic/engine

In [0]:
%run ./preborn_config

In [0]:
%run ./preborn_cohort

In [0]:
%run ./preborn_derive

In [0]:
import pandas as pd, numpy as np, json, os, uuid, traceback
from datetime import datetime, timezone
from pyspark.sql import types as T

dbutils.widgets.dropdown("role", "mother", ["mother", "infant", "derive"])
dbutils.widgets.dropdown("mode", "full", ["fixture", "full"])
dbutils.widgets.dropdown("stage", "pre", ["auto", "pre", "post"])
dbutils.widgets.text("cohort_persons", "")

role = dbutils.widgets.get("role")
mode = dbutils.widgets.get("mode")
stage = dbutils.widgets.get("stage")
_cp = dbutils.widgets.get("cohort_persons").strip()
cohort_persons = int(_cp) if _cp else None
if mode == "fixture" and cohort_persons is None:
    cohort_persons = 200

def _utcnow():
    return datetime.now(timezone.utc).isoformat()

print(f"role={role} mode={mode} stage={stage} cohort_persons={cohort_persons}")

In [0]:
# =====================================================================================
# Per-role driver: binds the active cfg + working schema and provides the trimmed
# assemble (person-rooted facts, optional base_rate infant_attr, no reference/death).
# =====================================================================================
def _cfg_for(role):
    return PREBORN_MOTHER_CONFIG if role == "mother" else PREBORN_INFANT_CONFIG

def _run_pre(cfg):
    tgt = f"{cfg.target['catalog']}.{cfg.target['schema']}"
    vol = cfg.target["volume"]
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {tgt}")
    os.makedirs(f"{vol}/config", exist_ok=True)
    n = build_preborn_cohort(cfg, n_persons=cohort_persons)
    run_id = uuid.uuid4().hex[:12]
    with open(f"{vol}/config/run_manifest.json", "w") as f:
        json.dump({"run_id": run_id, "created_utc": _utcnow(), "mode": mode,
                   "role": cfg.extra["role"], "cohort_persons": int(n),
                   "stage": "cohort_ready"}, f, indent=2, default=str)
    print(f"[{cfg.extra['role']}] PRE complete: cohort_person={n}, run_id={run_id}")
    return run_id

# ---- mock generation (generic over cfg.table_order; person-rooted) ----
def _mock_generate(cfg):
    tgt = f"{cfg.target['catalog']}.{cfg.target['schema']}"
    rng = np.random.default_rng(123)
    counts = {}
    order = topo_order(cfg)
    pkbase = pk_id_base(cfg)

    def _typed_write(name, pdf, schema_table):
        sch = spark.table(f"{tgt}.cohort_{name}").schema
        by = {f.name: f.dataType for f in sch.fields}
        out = pdf.copy()
        fields = []
        for c in out.columns:
            dt = by.get(c, T.StringType())
            if isinstance(dt, (T.IntegerType, T.LongType, T.ShortType, T.ByteType)):
                out[c] = pd.to_numeric(out[c], errors="coerce").astype("Int64")
            elif isinstance(dt, (T.FloatType, T.DoubleType)):
                out[c] = pd.to_numeric(out[c], errors="coerce").astype("float64")
            elif isinstance(dt, (T.DateType, T.TimestampType)):
                out[c] = pd.to_datetime(out[c], errors="coerce")
            else:
                dt = T.StringType()
                out[c] = out[c].astype("object").where(out[c].notna(), None)
            fields.append(T.StructField(c, dt, True))
        sdf = spark.createDataFrame(out, schema=T.StructType(fields))
        sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{tgt}.gen_{name}")
        counts[name] = int(len(out))
        print(f"  gen_{name}: {len(out)} rows")

    def _frame(name):
        allc = [f.name for f in spark.table(f"{tgt}.cohort_{name}").schema.fields]
        fcols = frame_columns(cfg, name, allc)
        return spark.table(f"{tgt}.cohort_{name}").toPandas()[fcols]

    def _shuffle(pdf, pk, base):
        if pk is None or len(pdf) == 0:
            return pdf.copy(), {}
        new = base + rng.permutation(len(pdf)) * 7 + 3
        old = pdf[pk].to_numpy()
        pdf = pdf.copy(); pdf[pk] = new
        return pdf, dict(zip(old, new))

    # subject (person)
    subj = cfg.subject
    pdf, pmap = _shuffle(_frame(subj), cfg.tables[subj].pk, pkbase[subj] + 50_000)
    _typed_write(subj, pdf, subj)

    for name in order:
        if name == subj:
            continue
        spec = cfg.tables[name]
        pdf = _frame(name)
        # remap the context fk (person_id) into the shuffled person keys
        cfk = spec.context_fk[0] if spec.context_fk else None
        if cfk and cfk in pdf.columns:
            pdf[cfk] = pdf[cfk].map(pmap)
            pdf = pdf[pdf[cfk].notna()].reset_index(drop=True)
        pdf, _ = _shuffle(pdf, spec.pk, pkbase[name] + 50_000)
        _typed_write(name, pdf, name)
    return counts

# ---- trimmed assemble: person-rooted facts, base_rate infant_attr, no reference ----
def _assemble(cfg, run_id):
    tgt = f"{cfg.target['catalog']}.{cfg.target['schema']}"
    subj = cfg.subject
    subj_pk = cfg.tables[subj].pk
    pkbase = pk_id_base(cfg)
    order = topo_order(cfg)

    def gen(name):
        return spark.table(f"{tgt}.gen_{name}").toPandas()

    def _src_schema(name):
        base = cfg.tables[name].source_table or name
        # cohort tables carry the full real column set for this table
        return spark.table(f"{tgt}.cohort_{name}").schema

    def _pairs(spec):
        return [(d, d + "time") for d in spec.date_cols if (d + "time") in spec.date_cols]

    def write_staged(name):
        pdf = frames[name]
        sch = _src_schema(name)
        names = [f.name for f in sch.fields]
        pdf = pdf.copy()
        for c in names:
            if c not in pdf.columns:
                pdf[c] = pd.NA
        # PII scrub
        for c in set(cfg.pii.get(name, [])) | set(cfg.pii.get(cfg.tables[name].source_table or name, [])):
            if c in names:
                pdf[c] = pd.NA
        pdf = pdf[names]
        # coerce to schema types
        for f in sch.fields:
            s = pdf[f.name]; dt = f.dataType
            if isinstance(dt, (T.IntegerType, T.LongType, T.ShortType, T.ByteType)):
                pdf[f.name] = pd.to_numeric(s, errors="coerce").astype("Int64")
            elif isinstance(dt, (T.FloatType, T.DoubleType)):
                pdf[f.name] = pd.to_numeric(s, errors="coerce").astype("float64")
            elif isinstance(dt, (T.DateType, T.TimestampType)):
                pdf[f.name] = pd.to_datetime(s, errors="coerce")
            else:
                pdf[f.name] = s.astype("object").where(s.notna(), None)
        nullable = T.StructType([T.StructField(f.name, f.dataType, True) for f in sch.fields])
        spark.createDataFrame(pdf, schema=nullable).write.mode("overwrite") \
            .option("overwriteSchema", "true").saveAsTable(f"{tgt}.stg_{name}")
        print(f"  staged stg_{name}: {len(pdf)} rows")

    frames = {}
    # subject
    subj_spec = cfg.tables[subj]
    sdf = gen(subj)
    subj_keyed, subj_map = rekey(sdf, subj_pk, pkbase[subj])
    subj_keyed = clamp_valid_range(subj_keyed, subj_spec.date_cols)
    frames[subj] = subj_keyed
    birth_col = (cfg.lifespan_rule or {}).get("birth_col")

    def _birth_floor(df, spec):
        if not birth_col or subj_pk not in df.columns or df.empty or not spec.date_cols:
            return df
        merged = df.merge(subj_keyed[[subj_pk, birth_col]], on=subj_pk, how="left", suffixes=("", "__b"))
        bname = birth_col if birth_col not in df.columns else birth_col + "__b"
        merged = birth_before_events(merged, bname, [c for c in spec.date_cols if c != bname])
        return merged.drop(columns=[bname])

    for name in order:
        if name == subj:
            continue
        spec = cfg.tables[name]
        df = gen(name)
        cfk = spec.context_fk[0]
        df = remap_fk(df, cfk, subj_map, subj_pk, how="inner")
        if spec.base_rate_table:
            df = df.drop_duplicates(subset=[cfk], keep="first").reset_index(drop=True)
        if spec.pk:
            df, _ = rekey(df, spec.pk, pkbase[name])
        for s_c, e_c in zip([c for c in spec.date_cols if "start" in c],
                            [c for c in spec.date_cols if "end" in c]):
            df = order_pair(df, s_c, e_c)
        df = _birth_floor(df, spec)
        for s_c, e_c in zip([c for c in spec.date_cols if "start" in c],
                            [c for c in spec.date_cols if "end" in c]):
            df = order_pair(df, s_c, e_c)
        if spec.self_fk and [c for c in spec.date_cols if "start" in c]:
            start0 = [c for c in spec.date_cols if "start" in c][0]
            df = rebuild_preceding_chain(df, cfk, spec.pk, start0, spec.self_fk)
        df = clamp_valid_range(df, spec.date_cols)
        df = date_from_datetime(df, _pairs(spec))
        frames[name] = df

    for name in order:
        write_staged(name)
    print(f"[{cfg.extra['role']}] ASSEMBLE complete: {len(order)} staged tables")


# =====================================================================================
# DISPATCH
# =====================================================================================
if role in ("mother", "infant"):
    cfg = _cfg_for(role)
    vol = cfg.target["volume"]
    if stage in ("auto", "pre"):
        run_id = _run_pre(cfg)
    if mode == "fixture" and stage in ("auto", "post"):
        print(f"[{role}] FIXTURE: mock generate -> assemble")
        counts = _mock_generate(cfg)
        run_id = json.load(open(f"{vol}/config/run_manifest.json"))["run_id"]
        _assemble(cfg, run_id)
        print(f"[{role}] fixture complete")
    elif mode == "full" and stage == "post":
        run_id = json.load(open(f"{vol}/config/run_manifest.json"))["run_id"]
        _assemble(cfg, run_id)
    elif mode == "full" and stage == "auto":
        print(f"[{role}] PRE done — now run preborn_train (GPU) for this role, then stage=post")

elif role == "derive":
    print("DERIVE: unify mother+infant staged base, re-derive deterministic tables, validate")
    derive_all()
    print("DERIVE complete — run preborn_tests for invariant validation")

dbutils.notebook.exit(json.dumps({"role": role, "mode": mode, "stage": stage}))